<h1>Data Ingestion and Training By: G5-Rita Group</h1><i>notebook edition</i><br>
1). Obtain<br>
2). Decompress<br>
3). Parse<br>
<li>
<ol>A) Load Game</ol>
<ol>B) Parse Move</ol>
<ol>C) Load Move</ol>
</li>
4). Cleanup

In [1]:
#Define connection to MYSQL Server
ConnectString = "mysql -h database-1.clqxqhhe6wft.us-east-1.rds.amazonaws.com -P 3306 -u admin -p'<Enter_DB_Password>' --ssl-verify-server-cert  --ssl-ca=/certs/global-bundle.pem mysql"

host=ConnectString[9:60]
user='admin'
password='Data608-Project'
database='CHESSBOT'


In [2]:
#Define file to load into Database

#Provide two-digit month   (01 -> 12)
File_month = "01"

#Provide four-digit year  (2026  <- 2013 )
File_year = "2013"


In [3]:
####   OBTAIN   the file using bash commands

import wget

CompressedFileName = "lichess_db_standard_rated_" + File_year + "-" + File_month + ".pgn.zst"
CompressedCompletePath = "https://database.lichess.org/standard/" + CompressedFileName
EBSPath = "/data"
UnCompressedFile = CompressedFileName[:-4]
UnCompressedCompletePath = f"{EBSPath}/{UnCompressedFile}"

# Download the file
DownloadedCompressedFile = wget.download(CompressedCompletePath, out=EBSPath)
print()
print("File saved to: " + DownloadedCompressedFile)


100% [........................................................] 17761302 / 17761302
File saved to: /data/lichess_db_standard_rated_2013-01.pgn.zst


In [4]:
####   DECOMPRESS the file using bash commands
import zstandard as zstd

dctx = zstd.ZstdDecompressor()

with open(DownloadedCompressedFile, 'rb') as ifh, open(UnCompressedCompletePath, 'wb') as ofh:
    # copy_stream handles the heavy lifting of reading/writing chunks
    dctx.copy_stream(ifh, ofh)
    
print(f"Decompression finished: {UnCompressedCompletePath}")


Decompression finished: /data/lichess_db_standard_rated_2013-01.pgn


In [ ]:
### PARSE the uncompressed file
import mysql.connector
import parse as pa
metric = []
successbit = False

#connect to SQL server
print("Attempting MySQL Server connection")
connection = mysql.connector.connect(host=host,user=user,password=password,database=database,autocommit=True) 

if connection.is_connected():
    cursor = connection.cursor()
    print("Connected to MySQL Server: ")
    metric,successbit = pa.Parse(UnCompressedCompletePath,cursor)
    print("Total Events Ingested: ",metric[0])
    print("Total Events Considered: ",metric[1])
    print("Log results: ",metric[2])

else:
    print("Failed connection to MySQL Server")
    

Attempting MySQL Server connection
Connected to MySQL Server: 
LinesTotalRead:  10000
LinesTotalRead:  20000
LinesTotalRead:  30000
LinesTotalRead:  40000
LinesTotalRead:  50000
LinesTotalRead:  60000
LinesTotalRead:  70000
LinesTotalRead:  80000
LinesTotalRead:  90000
LinesTotalRead:  100000
LinesTotalRead:  110000
LinesTotalRead:  120000
LinesTotalRead:  130000
LinesTotalRead:  140000
LinesTotalRead:  150000
LinesTotalRead:  160000


In [ ]:
### CLEANUP  the drive

import cleanup as cu
print("Starting Cleanup:")
storageRemain = cu.Cleanup("",DownloadedCompressedFile,UnCompressedCompletePath)
print("Output storageRemain: ",storageRemain)
print("Starting Cleanup:")